In [11]:
import re
import numpy as np
import pandas as pd

# CSV_PATH : 우리가 불러올 리뷰 데이터 파일 경로
# TEXT_COL : 리뷰 본문이 들어 있는 컬럼명
CSV_PATH = "../all/cleaned_reviews.csv"
TEXT_COL = "리뷰내용"

df = pd.read_csv(CSV_PATH)
# 리뷰 텍스트(review)가 비어 있는 행은 분석에 사용할 수 없으므로 제거한다.
# dropna(subset=[TEXT_COL]) : review 컬럼이 NaN인 행만 제거
# reset_index(drop=True)    : 제거 후 인덱스를 0부터 다시 정리
df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)

def clean_text(text):
	# 혹시 숫자나 NaN 등이 들어와도 문자열로 처리 가능하게 변환
    text = str(text)

    # 영어 소문자 변환
    text = text.lower()
    
    # HTML태그/URL/이메일 제거
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    
    # 줄바꿈(\r, \n), 탭(\t)을 공백으로 치환
    text = re.sub(r"[\r\n\t]+", " ", text)

    # 이모지 및 특수 문구 제거 (한글, 영문, 숫자, 기본 문장부호만 남김)
    # 아래 정규식은 한글, 영문, 숫자, 공백, 그리고 . , ! ? ~ 를 제외한 모든 것을 제거합니다.
    text = re.sub(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-z0-9\s\.\,\!\?\~]", " ", text)
    
    # 의미 없는 반복 문자/단어 축소
    # 2글자 이상 단어/구절 반복 축소 (예: 좋아요좋아요좋아요 -> 좋아요좋아요)
    text = re.sub(r'(.{2,}?)\1{2,}', r'\1\1', text)
    # 1글자 단일 문자 반복 축소 (예: ㅋㅋㅋㅋ -> ㅋㅋ, !!!! -> !!)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 공백이 여러 개 연속으로 있으면 하나의 공백으로 축소
    # 그리고 문자열 양끝의 공백 제거
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_review"] = df[TEXT_COL].apply(clean_text)
# 길이가 너무 짧은 리뷰는 토픽 모델링에 거의 도움이 안 될 수 있다.
# 여기서는 길이 3 이상만 남긴다.
df = df[df["clean_review"].str.len() >= 3].reset_index(drop=True)

# BERTopic에는 최종적으로 문서 리스트 형태로 텍스트를 넣기 때문에, 파이썬 리스트로 변환
docs = df["clean_review"].tolist()

print("리뷰 수:", len(docs))
display(df.head(30))

리뷰 수: 547359


,리뷰번호,goodsNo,작성자,리뷰내용,평점,체험단,구매옵션,키,몸무게,성별,...,구매상세,연도,월,일,요일,평점_raw,노출일수,일평균_도움돼요지수,도움여부,clean_review
0,441368,234480,-,"174/64 엠사이즈 딱 잘맞습니다.\n\n허리 고무줄이라 편하구요, 스판 엄청 ...",5.0,False,블랙M(30~32),176.0,76.0,missing,...,블랙M(~),2015,12,21,월,5,3754,0.00000,0,"174 64 엠사이즈 딱 잘맞습니다. 허리 고무줄이라 편하구요, 스판 엄청 신축성좋..."
1,2113020,234480,-,"178cm, 66kg.\n길이는 좀 짧고 더워서 가을에 입어야할 듯.",3.0,False,블랙 · M,170.0,61.0,missing,...,블랙,2018,5,18,금,3,2875,0.00000,0,"178cm, 66kg. 길이는 좀 짧고 더워서 가을에 입어야할 듯."
2,3841497,234480,-,"검은색으로 삿는데 사이즈, 디자인 괜찮고 되게 편해요 ㅎㅎ",5.0,False,블랙 · L,176.0,76.0,missing,...,블랙,2019,2,6,수,5,2611,0.00000,0,"검은색으로 삿는데 사이즈, 디자인 괜찮고 되게 편해요 ㅎㅎ"
3,381540,234480,-,고무줄 바지가 편해서 벌써 제멋에서 3번째 구매하고 있습니다 ㅎㅎ \n세탁해도 크게...,5.0,False,블랙XL(34~36),176.0,76.0,missing,...,블랙XL(~),2015,11,8,일,5,3797,0.00000,0,고무줄 바지가 편해서 벌써 제멋에서 3번째 구매하고 있습니다 ㅎㅎ 세탁해도 크게 변...
4,11874438,234480,-,기장이 좀 짧은것 같습니다. 나머지는 여유있고 좋아요. 굳,5.0,False,블랙 · M,170.0,61.0,missing,...,블랙,2020,9,25,금,5,2014,0.00000,0,기장이 좀 짧은것 같습니다. 나머지는 여유있고 좋아요. 굳
5,399430,234480,-,딱맞으<br />\n바지도 편하고 딷듯하고 대만족임\n엠사이즈도 생각보다크니까 ㅇ엠...,5.0,False,차콜M(30~32),176.0,76.0,missing,...,차콜M(~),2015,11,28,토,5,3777,0.00000,0,딱맞으 바지도 편하고 딷듯하고 대만족임 엠사이즈도 생각보다크니까 ㅇ엠사이즈입으세용 ...
6,492945,234480,-,라지 사이즈 구매하였는대 핏이 정말 좋습니다.\n다음에도 주문할게요,5.0,False,블랙L(32~34),176.0,76.0,missing,...,블랙L(~),2016,2,4,목,5,3709,0.00000,0,라지 사이즈 구매하였는대 핏이 정말 좋습니다. 다음에도 주문할게요
7,489016,234480,-,배기핏인지 사놓고 알았네요 ㅋㅋㅋ 제일작게사긴했는데 너무커서 껴입을때만 입게되네요,5.0,False,블랙M(30~32),176.0,76.0,missing,...,블랙M(~),2016,1,31,일,5,3713,0.00000,0,배기핏인지 사놓고 알았네요 ㅋㅋ 제일작게사긴했는데 너무커서 껴입을때만 입게되네요
8,489017,234480,-,배기핏인지 사놓고 알았네요 ㅋㅋㅋ 제일작게사긴했는데 너무커서 껴입을때만 입게되네요,5.0,False,블랙M(30~32),176.0,76.0,missing,...,블랙M(~),2016,1,31,일,5,3713,0.00000,0,배기핏인지 사놓고 알았네요 ㅋㅋ 제일작게사긴했는데 너무커서 껴입을때만 입게되네요
9,4393000,234480,-,블랙이라기보단 회녹색 편하고 잘입고있음 저렴하구요,5.0,False,차콜 · M,170.0,61.0,missing,...,차콜,2019,4,3,수,5,2555,0.00000,0,블랙이라기보단 회녹색 편하고 잘입고있음 저렴하구요


In [12]:
from konlpy.tag import Okt

# Okt 객체 생성
# Okt는 KoNLPy에서 많이 쓰는 한국어 형태소 분석기
okt = Okt()

# 불용어(stopwords) 정의
# 너무 자주 나오지만 토픽을 구분하는 데 도움을 덜 주는 단어들
# 예: 하다, 되다, 있다, 너무, 그냥, 제품, 상품 등
custom_stopwords = {
    "하다", "되다", "있다", "없다", "이다", "같다",
    "정말", "진짜", "너무", "아주", "매우", "조금", "좀",
    "제품", "상품", "구매", "사용", "이번", "그냥"
}

# 한국어 리뷰 문장을 토큰(단어) 리스트로 변환하는 함수
def tokenize_ko(text):
    tokens = []
		
		# okt.pos(text, norm=True, stem=True)
    # - norm=True : 구어체/반복 표현 등을 어느 정도 정규화
    # - stem=True : 활용형을 기본형에 가깝게 변환
    #
    # 반환 예시:
    # [("배송", "Noun"), ("늦다", "Adjective"), ...]
    for word, pos in okt.pos(text, norm=True, stem=True):
		    # 우리가 토픽 해석에 주로 쓰고 싶은 품사만 남긴다.
        # Noun      : 명사
        # Verb      : 동사
        # Adjective : 형용사
        # Alpha     : 영문 토큰(예: app, ios 등)
        if pos in {"Noun", "Verb", "Adjective", "Alpha"}:
		        # 길이 1인 토큰은 의미가 약한 경우가 많아 제거
            # 또한 직접 정의한 불용어도 제외
            if len(word) > 1 and word not in custom_stopwords:
                tokens.append(word)

    return tokens